In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Leakage-free fold generator for Mozilla bug severity/priority data.

Generates 10-fold stratified CV splits from the raw, unbalanced Mozilla
bug reports.

No leakage:
  - TF-IDF fit on train only, test is transform-only.
  - One-hot scheme fixed from full data (label-independent, so it's fine).
  - SMOTE/undersampling: train only. Test always stays in its real,
    imbalanced distribution.

Writes Starlang .data files per fold/condition: comma-separated, label
last column, no header, CRLF.

File naming:
  {Product}_{Target}{XfeatSuffix}_{condition}_fold{i}_train.data
  {Product}_{Target}{XfeatSuffix}_fold{i}_test.data   (shared across conditions)

Add new products to PRODUCTS below.
"""

import os
import re
import sys
import argparse
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler



# CONFIG

CONFIG = {
    "raw_csv": "Mozilla_complete_dataset.csv",
    "raw_encoding": "latin-1",          
    "out_dir": "folds_output",

    "n_splits": 10,
    "random_state": 42,

    "tfidf_max_features": 100,
    "min_summary_words": 5,              

    # products to generate folds for (exact 'product' column value)
    "products": ["Core", "Firefox"],

    "targets": {
        "Priority": {
            "label_col": "priority",
            "valid_classes": ["P1", "P2", "P3", "P4", "P5"],   
            "cross_feature": "severity",
        },
        "Severity": {
            "label_col": "severity",
            "valid_classes": ["critical", "normal", "S3", "S4"],
            "cross_feature": "priority",
        },
    },

    "base_cat_cols": ["component", "type"],
    "text_col": "summary",

    "conditions": ["processed", "smote", "undersampled"],

    # withcross = cross-field feature included, nocross = excluded
    "cross_variants": {
        "withcross": True,
        "nocross": False,
    },
}

# HELPERS

def get_stopwords():
    nltk.download("stopwords", quiet=True)
    return set(stopwords.words("english"))


_STOP = None
def clean_text(text):
    global _STOP
    if _STOP is None:
        _STOP = get_stopwords()
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    words = [w for w in text.split() if w not in _STOP]
    return " ".join(words)


def load_and_clean(cfg):
    """Load raw data + deterministic cleaning. Safe to run before the fold
    split since none of this looks at labels."""
    path = cfg["raw_csv"]
    print(f"[1] Loading raw data: {path}")
    df = pd.read_csv(path, encoding=cfg["raw_encoding"], low_memory=False)
    df.columns = [c.strip().lower() for c in df.columns]

    df.replace(["--", "-"], np.nan, inplace=True)

    drop_cols = ["id", "keywords", "status", "blocks", "depends_on"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    print("[2] Cleaning text (lowercase, punctuation, stopwords, min length)...")
    df[cfg["text_col"]] = df[cfg["text_col"]].apply(clean_text)
    df = df[df[cfg["text_col"]].apply(lambda x: len(x.split()) >= cfg["min_summary_words"])]

    df = df.reset_index(drop=True)
    print(f"    Rows remaining: {len(df)}")
    return df


def subset_for_task(df, product, target_cfg, cross_feature_needed):
    """Subset for one product+target, filter to valid classes, drop rows
    missing required columns."""
    label_col = target_cfg["label_col"]
    valid = target_cfg["valid_classes"]

    sub = df[df["product"] == product].copy()
    sub = sub[sub[label_col].astype(str).str.strip().isin(valid)]

    needed = [CONFIG["text_col"]] + CONFIG["base_cat_cols"] + [label_col]
    if cross_feature_needed:
        needed.append(target_cfg["cross_feature"])
    needed = list(dict.fromkeys(needed))

    sub = sub.dropna(subset=needed)
    sub = sub.reset_index(drop=True)
    return sub


def build_fixed_onehot(df_all_for_product, cat_cols):
    """Fix the one-hot scheme on the full data to keep column count constant across folds. Not refit per
    fold, unlike TF-IDF."""
    enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore", dtype=np.float64)
    enc.fit(df_all_for_product[cat_cols].astype(str))
    return enc


def to_starlang_lines(X, y):
    """X, y -> Starlang .data lines (comma-separated, label last)."""
    lines = []
    for row, lab in zip(X, y):
        feats = ",".join(f"{v:.6g}" for v in row)
        lines.append(f"{feats},{lab}")
    return lines


def write_data_file(path, X, y):
    lines = to_starlang_lines(X, y)
    # CRLF, incl. trailing, to match existing files
    with open(path, "w", newline="") as f:
        f.write("\r\n".join(lines) + "\r\n")


# MAIN LOOP

def generate_for_task(df_clean, product, target_name, cfg):
    target_cfg = cfg["targets"][target_name]
    label_col = target_cfg["label_col"]
    cross_col = target_cfg["cross_feature"]
    text_col = cfg["text_col"]

    print("\n" + "=" * 64)
    print(f"  PRODUCT={product}  TARGET={target_name}")
    print("=" * 64)

    # build widest subset once (incl. cross-feature); both variants derive
    # from it so the test folds match
    sub = subset_for_task(df_clean, product, target_cfg, cross_feature_needed=True)
    if len(sub) == 0:
        print("  (no data for this product/target, skipping)")
        return
    print(f"  Subset size: {len(sub)}")
    print(f"  Class distribution:\n{sub[label_col].value_counts().to_string()}")

    y_all = sub[label_col].astype(str).str.strip().values

    skf = StratifiedKFold(n_splits=cfg["n_splits"], shuffle=True,
                          random_state=cfg["random_state"])

    for cross_suffix, include_cross in cfg["cross_variants"].items():
        cat_cols = list(cfg["base_cat_cols"])
        if include_cross:
            cat_cols = cat_cols + [cross_col]

        onehot = build_fixed_onehot(sub, cat_cols)
        n_cat_features = len(onehot.get_feature_names_out(cat_cols))
        n_total_features = cfg["tfidf_max_features"] + n_cat_features

        out_base = f"{product}_{target_name}_{cross_suffix}"
        print(f"\n  -- Variant: {cross_suffix} "
              f"(cross-feature {'INCLUDED' if include_cross else 'EXCLUDED'}) | "
              f"features={n_total_features} ({cfg['tfidf_max_features']} tfidf + {n_cat_features} one-hot)")

        for fold_i, (tr_idx, te_idx) in enumerate(skf.split(sub[text_col], y_all)):
            tr = sub.iloc[tr_idx]
            te = sub.iloc[te_idx]

            # TF-IDF: fit on train only, transform test
            vec = TfidfVectorizer(max_features=cfg["tfidf_max_features"])
            Xtr_tfidf = vec.fit_transform(tr[text_col]).toarray()
            Xte_tfidf = vec.transform(te[text_col]).toarray()

            # train TF-IDF can yield <max_features cols (rare); pad with
            # zeros, Java side needs a fixed schema
            if Xtr_tfidf.shape[1] < cfg["tfidf_max_features"]:
                pad = cfg["tfidf_max_features"] - Xtr_tfidf.shape[1]
                Xtr_tfidf = np.hstack([Xtr_tfidf, np.zeros((Xtr_tfidf.shape[0], pad))])
                Xte_tfidf = np.hstack([Xte_tfidf, np.zeros((Xte_tfidf.shape[0], pad))])

            Xtr_cat = onehot.transform(tr[cat_cols].astype(str))
            Xte_cat = onehot.transform(te[cat_cols].astype(str))

            Xtr = np.hstack([Xtr_tfidf, Xtr_cat])
            Xte = np.hstack([Xte_tfidf, Xte_cat])
            ytr = tr[label_col].astype(str).str.strip().values
            yte = te[label_col].astype(str).str.strip().values

            # test file is shared across conditions, untouched real distribution
            test_path = os.path.join(cfg["out_dir"], f"{out_base}_fold{fold_i}_test.data")
            write_data_file(test_path, Xte, yte)

            # one train file per balancing condition
            for cond in cfg["conditions"]:
                if cond == "processed":
                    Xc, yc = Xtr, ytr
                elif cond == "smote":
                    sm = SMOTE(random_state=cfg["random_state"])
                    Xc, yc = sm.fit_resample(Xtr, ytr)
                elif cond == "undersampled":
                    rus = RandomUnderSampler(random_state=cfg["random_state"])
                    Xc, yc = rus.fit_resample(Xtr, ytr)
                else:
                    raise ValueError(f"unknown condition: {cond}")

                train_path = os.path.join(
                    cfg["out_dir"], f"{out_base}_{cond}_fold{fold_i}_train.data")
                write_data_file(train_path, Xc, yc)

        print(f"     Wrote {cfg['n_splits']} folds x {len(cfg['conditions'])} conditions "
              f"+ {cfg['n_splits']} shared test files -> prefix '{out_base}'")


def main():
    parser = argparse.ArgumentParser(description="Leakage-free fold generator")
    parser.add_argument("--raw", default=CONFIG["raw_csv"], help="path to raw CSV")
    parser.add_argument("--out", default=CONFIG["out_dir"], help="output directory")
    parser.add_argument("--products", nargs="*", default=None,
                        help="products to process (default: CONFIG)")
    parser.add_argument("--targets", nargs="*", default=None,
                        help="targets to process: Priority Severity")
    
    args, _unknown = parser.parse_known_args()

    cfg = dict(CONFIG)
    cfg["raw_csv"] = args.raw
    cfg["out_dir"] = args.out
    if args.products:
        cfg["products"] = args.products

    os.makedirs(cfg["out_dir"], exist_ok=True)

    df_clean = load_and_clean(cfg)

    targets = args.targets if args.targets else list(cfg["targets"].keys())
    for product in cfg["products"]:
        for target_name in targets:
            generate_for_task(df_clean, product, target_name, cfg)

    print("\n[OK] All fold files generated ->", cfg["out_dir"])


if __name__ == "__main__":
    main()